In [ ]:
import sys
!{sys.executable} -m pip install plotly numpy
import pandas as pd
import sqlite3
import plotly.graph_objects as go
import numpy as np

## Plotly (animated chart)

Goal:
Create an **animated** version of the "Dynamic of commits per user in project1" chart.

Steps:
1) Load `uid`, `timestamp`, `numTrials` from `checker` for `project1` where `status = "ready"`.
2) Aggregate to daily max progress per user (max `numTrials` per day).
3) Forward-fill missing days so lines don't "drop" to zero.
4) Build a Plotly `go.Figure` with `frames` and a `play` button.

In [ ]:
import pandas as pd
import sqlite3
import plotly.graph_objects as go
import numpy as np

# Connect to SQLite (we are in src/ex09, so ../data is correct)
con = sqlite3.connect("../data/checking-logs.sqlite")

# Load progress logs for project1 (users only, status='ready')
q = """
SELECT uid, timestamp, numTrials
FROM checker
WHERE uid LIKE 'user_%'
  AND labname = 'project1'
  AND status = 'ready';
"""
raw = pd.read_sql(q, con, parse_dates=["timestamp"])
con.close()

raw = raw.sort_values(["uid", "timestamp"])
raw.head()

In [ ]:
# Convert timestamp to date (one point per day per user)
raw["date"] = raw["timestamp"].dt.date

# Daily max numTrials per user (progress by day)
daily = (
    raw.groupby(["date", "uid"])["numTrials"]
       .max()
       .unstack("uid")
       .sort_index()
)

# Forward fill: once a user reaches X commits, they keep that value until they improve
daily = daily.ffill().fillna(0)

# Optional: sort users by final progress (leader appears first in legend)
final_rank = daily.iloc[-1].sort_values(ascending=False)
daily = daily[final_rank.index]

daily.head()

In [ ]:
# We use "day index" on x-axis (1..N) like the reference lineRace.py
num_frames = daily.shape[0]
xaxis_range = [0, num_frames + 2]

# Initial state (first day)
x_init = np.array([1])

initial_data = []
for uid in daily.columns:
    y0 = np.array([daily.loc[daily.index[0], uid]])
    initial_data.append(
        go.Scatter(
            x=x_init,
            y=y0,
            mode="lines+markers",
            name=uid
        )
    )

# Frames: each frame reveals data up to day f
frames = []
dates = list(daily.index)

for f in range(1, num_frames + 1):
    x_axis = np.arange(1, f + 1)
    curr_data = []
    for uid in daily.columns:
        y_axis = daily[uid].values[:f]
        curr_data.append(
            go.Scatter(
                x=x_axis,
                y=y_axis,
                mode="lines+markers",
                name=uid
            )
        )

    title = f"Dynamic of commits per user in project1 (day {f}/{num_frames}) - {dates[f-1]}"
    frames.append(go.Frame(data=curr_data, layout={"title": title}))

In [ ]:
fig = go.Figure(
    data=initial_data,
    layout={
        "title": "Dynamic of commits per user in project1",
        "xaxis": {"range": xaxis_range, "title": "day"},
        "yaxis": {"title": "numTrials"},
        "template": "simple_white",
        "updatemenus": [
            {
                "type": "buttons",
                "buttons": [
                    {
                        "method": "animate",
                        "label": "play",
                        "args": [None, {"frame": {"duration": 200, "redraw": True}, "fromcurrent": True}]
                    }
                ],
            }
        ],
    },
    frames=frames,
)

fig.show()